# AI-Powered Healthcare Diagnosis Assistant

**Author: Vikas Yadav**

End-to-end ML project for classifying breast tumors as malignant or benign using the Wisconsin Diagnostic Breast Cancer dataset.

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
RANDOM_STATE=42


## 1. Load and explore the dataset

In [ ]:
data=load_breast_cancer()
X=pd.DataFrame(data.data,columns=data.feature_names)
y=pd.Series(data.target,name="diagnosis")
df=X.copy(); df["diagnosis"]=y.map({0:"malignant",1:"benign"})
print("Dataset shape:",X.shape)
print(df["diagnosis"].value_counts())
df.head()


## 2. Preprocess and split

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=RANDOM_STATE)


## 3. Compare five ML models

In [ ]:
models={"Logistic Regression":LogisticRegression(max_iter=5000),"Random Forest":RandomForestClassifier(n_estimators=300,random_state=42),"Gradient Boosting":GradientBoostingClassifier(random_state=42),"SVM":SVC(probability=True),"KNN":KNeighborsClassifier()}
results=[]
scaler=StandardScaler(); Xtr=scaler.fit_transform(X_train); Xte=scaler.transform(X_test)
for name,m in models.items():
 m.fit(Xtr,y_train); pred=m.predict(Xte); results.append([name,accuracy_score(y_test,pred),precision_score(y_test,pred),recall_score(y_test,pred),f1_score(y_test,pred)])
pd.DataFrame(results,columns=["Model","Accuracy","Precision","Recall","F1"]).sort_values("Accuracy",ascending=False)


## 4. Tune the model with GridSearchCV

In [ ]:
pipe=Pipeline([("scaler",StandardScaler()),("model",LogisticRegression(max_iter=5000,random_state=42))])
grid=GridSearchCV(pipe,{"model__C":[.01,.1,1,10,100]},cv=5,scoring="accuracy",n_jobs=-1)
grid.fit(X_train,y_train); best_model=grid.best_estimator_
print("Best parameters:",grid.best_params_)


## 5. Evaluate and save

In [ ]:
pred=best_model.predict(X_test); proba=best_model.predict_proba(X_test)[:,1]
print("Accuracy:",accuracy_score(y_test,pred)); print("Precision:",precision_score(y_test,pred)); print("Recall:",recall_score(y_test,pred)); print("F1:",f1_score(y_test,pred)); print("ROC-AUC:",roc_auc_score(y_test,proba))
print(classification_report(y_test,pred,target_names=data.target_names))
joblib.dump(best_model,"healthcare_diagnosis_model.joblib")
print("Model saved successfully.")
